In [7]:
import datetime
import json
import os
import time
from multiprocessing import Pool, cpu_count

import ipywidgets as widgets
import numpy as np
import pandas as pd
import requests
from IPython.display import HTML, Markdown, display
from requests.auth import HTTPBasicAuth
from smart_services import SmartServices
from suds.client import Client
from Add_In_Queries import *

In [8]:
def display_scrollable_df(df, max_height="50vh", max_width="90vw"):
    style = f"""
    <div style="
        display: flex;
        justify-content: center;
        padding: 20px;
    ">
        <div style="
            overflow: auto;
            max-height: {max_height};
            max-width: {max_width};
            width: 100%;
            border: 1px solid #444;
            padding: 10px;
            background-color: #000;
            color: #eee;
            font-family: 'Arial Narrow', Arial, sans-serif;
            box-sizing: border-box;
        ">
            {df.to_html(classes='table', border=0, index=True)}
        </div>
    </div>
    """
    return HTML(style)

In [13]:
def get_pra(date,alto_smartservices):

    results = alto_smartservices.get(
        query_runner="PRA_all_Fields_scopes",
        params={"format": "json", "PRA_Start_Date": date, "PRA_end_historical": date},
    )

    return results


def safe_call_pra(tasks):
    func, date,sessions = tasks

    try:
        return date, func(date,sessions)
    except Exception as e:
        return {date: str(e), "args": tasks}
    

In [14]:
def display_ex_ante_app(sessions):

    
    funds=[]
    fields=[]

    date_start = widgets.Text(
        step=1,
        description="Start (YYYY-MM-DD)",
        disabled=False,
        display="flex",
        flex_flow="column",
        align_items="stretch",
        style={"description_width": "auto"},
    )

    end_date = widgets.Text(
        step=1,
        description="End (YYYY-MM-DD)",
        disabled=False,
        display="flex",
        flex_flow="column",
        align_items="stretch",
        style={"description_width": "auto"},
    )

    def get_data_on_click(b):

        global dataframe, tables, fields,funds

        start = date_start.value
        end = end_date.value

        date_list = pd.date_range(start, end, freq="W-FRI").strftime("%Y-%m-%d")

        # date_list=pd.bdate_range(start, end, freq='W-MON').strftime('%Y-%m-%d')
        tasks = [(get_pra, date,sessions) for date in date_list]

        dico_data = {}
        start = time.time()
        with Pool(processes=min(5, cpu_count() * 2)) as pool:
            for date, data in pool.imap_unordered(safe_call_pra, tasks):
                if type(data) is not str:
                    dico_data[date] = data
                    print(f'data found at {date}')
                else:
                    print(date)

        dataframe = pd.DataFrame()
        for date in dico_data:
            dataframe = pd.concat([dataframe, dico_data[date]])

        tables = {}
        AIL = dataframe[dataframe["COMP_LABEL"] == "AMUNDI.IRL"].copy()
        AIL["REPORT_DATE"] = pd.to_datetime(
            AIL["REPORT_DATE"], errors="coerce", utc=True
        )
        AIL["REPORT_DATE"] = AIL["REPORT_DATE"].apply(lambda x: x.replace(tzinfo=None))
        list_of_funds = set(AIL["SELECTEDFUNDCODE"])
        tables["Scope"] = pd.DataFrame(list_of_funds, columns=["Scope"])
        tables["Data"] = AIL
        tables["Fields"] = pd.DataFrame(AIL.columns, columns=["Ex Ante Fields"])

        funds=tables["Scope"]["Scope"]
        fields=tables["Fields"]["Ex Ante Fields"]

        dropdown1.options = fields
        dropdown2.options = funds

        def get_excel(b):

            with pd.ExcelWriter("Ex Ante Data.xlsx", engine="openpyxl") as writer:

                tables["Data"].to_excel(writer, sheet_name="Data", index=True)
                tables["Scope"].to_excel(writer, sheet_name="Scope", index=True)
                tables["Fields"].to_excel(writer, sheet_name="Scope", index=True)

            print("File Generated")

        bt_excel = widgets.Button(
            description="Get Excel",
            layout=widgets.Layout(
                display="flex",
                justify_content="center",
                align_items="center",
                spacing="10px",
                width="auto",
            ),
        )

        bt_excel.on_click(get_excel)
        with data_output:
            data_output.clear_output()
            display(display_scrollable_df(tables["Scope"]))
            display(display_scrollable_df(tables["Fields"]))
            # display(display_scrollable_df(get_perf_catalog()))
            display(bt_excel)
            # display(display_scrollable_df(tables['Data']))


    dropdown1 = widgets.Dropdown(description="Fields:", value=None, options=fields)
    dropdown2 = widgets.Dropdown(description="Funds:", value=None, options=funds)

    data_output = widgets.Output()
    button_data = widgets.Button(description="Get Data")
    button_data.on_click(get_data_on_click)

    parameters_ui = widgets.VBox(
        [
            widgets.HBox(
                [date_start, end_date, button_data],
                layout=widgets.Layout(
                    display="flex",
                    justify_content="center",
                    align_items="center",
                    spacing="auto",
                    width="auto",
                ),
            ),
            data_output,
        ]
    )

    data = []


    def on_add_constraint_clicked(b):
        row = {"Field": dropdown1.value, "Fund": dropdown2.value}
        data.append(row)
        with constraint_output:
            constraint_output.clear_output()
            display(pd.DataFrame(data))


    add_constraint_btn = widgets.Button(description="Add Filter", button_style="success")
    add_constraint_btn.on_click(on_add_constraint_clicked)

    constraint_output = widgets.Output()
    output = widgets.Output()


    def on_clear_clicked(b):
        data.clear()
        res.clear()
        with constraint_output:
            constraint_output.clear_output()
            display(pd.DataFrame(columns=["Field", "Fund"]))

        with output:
            output.clear_output()


    clear_btn = widgets.Button(description="Clear All", button_style="danger")
    clear_btn.on_click(on_clear_clicked)

    res = {}


    def on_optimize_clicked(b):

        filter_dataframe = pd.DataFrame(data)
        unique_list_funds = set(filter_dataframe["Fund"])
        dico_filter = {}
        for fund in unique_list_funds:
            temp = filter_dataframe[filter_dataframe["Fund"] == fund]
            dico_filter[fund] = list(set(temp["Field"]))

        for key in dico_filter:

            temp = tables["Data"][tables["Data"]["SELECTEDFUNDCODE"] == key].set_index(
                "REPORT_DATE"
            )[dico_filter[key]]
            res[key] = temp

        with output:
            output.clear_output()
            for key in res:
                display(Markdown("### " + str(key)))
                display(display_scrollable_df(res[key]))


    optimize_btn = widgets.Button(description="Filter", button_style="primary")
    optimize_btn.on_click(on_optimize_clicked)

    constraint_ui = widgets.VBox(
    [
        widgets.VBox([dropdown1, dropdown2]),
        widgets.HBox([add_constraint_btn, clear_btn, optimize_btn]),
        constraint_output,
        output,
    ]
        )

    tab_contents = ["Control", "Analysis"]

    children = [parameters_ui, constraint_ui]
    tab = widgets.Tab()
    tab.children = children
    for i, title in enumerate(tab_contents):
        tab.set_title(i, title)

    display(tab)

In [15]:
session = requests.session()  # Set the session request
session.stream = True
session.auth = HTTPBasicAuth(username='Guicharj', password='PioneerAmundi123')
session.verify = False

alto_smartservices = SmartServices(
base="https://smartservices-risk.intramundi.com/sms-ws/api/query-runner/",
requests_session=session)

display_ex_ante_app(alto_smartservices)